# DAX Credit Stress · Data Layer Build

**Dieses Notebook lädt alle Rohdaten für das Credit-Stress-Modell.**  

Führe die Zellen einzeln oder `Run All` aus. Jedes Modul cached in Parquet — beim zweiten Lauf geht alles in Millisekunden.

**Voraussetzungen:**
- Alle `0X_fetch_*.py` Dateien liegen im selben Ordner wie dieses Notebook
- `config.py` liegt ebenfalls daneben
- Für Svensson: Bundesbank-CSV liegt unter `data/bundesbank_svensson.csv`

## 0 · Setup

In [ ]:
import sys, os, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

# Sicherstellen, dass wir im richtigen Ordner sind
os.chdir(Path(__file__).parent if '__file__' in globals() else '.')
sys.path.insert(0, str(Path.cwd()))

# Erstinstallation, falls nötig:
# !pip install yfinance pandas pyarrow scipy openpyxl matplotlib

from config import show_config
show_config()

## 1 · DAX-40 Aktienkurse

6 Jahre tägliche Adj-Close-Kurse aller DAX-Mitglieder.

In [ ]:
%run 01_fetch_dax40_prices.py

In [ ]:
# Plot: normalisierte Performance
normalized = dax_prices.div(dax_prices.iloc[0]).dropna(how='all')
normalized.plot(figsize=(12, 5), legend=False, alpha=0.6, lw=0.8,
                title='DAX-40 · Normalisierte Performance (Basis=1.0)')
plt.grid(alpha=0.3)
plt.show()

## 2 · DAX-40 Bilanzdaten

Market Cap, Total Debt, kurz-/langfristige Verbindlichkeiten — aus yfinance Balance Sheets.

In [ ]:
%run 02_fetch_dax40_fundamentals.py

In [ ]:
# Check: welche Firmen haben brauchbare Debt-Daten?
ok = dax_fundamentals['TotalDebt'].notna().sum()
print(f'Total Debt verfügbar: {ok}/{len(dax_fundamentals)}')
missing = dax_fundamentals[dax_fundamentals['TotalDebt'].isna()][['Name']]
if len(missing) > 0:
    print('\nFehlend:')
    display(missing)

## 3 · Brent Crude — Energy-Faktor

In [ ]:
%run 03_fetch_brent_crude.py

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(brent_prices.index, brent_prices['Close_USD'], color='#c4341e', lw=1)
ax1.set_title('Brent Crude · USD/bbl')
ax1.grid(alpha=0.3)
ax2.plot(brent_prices.index, brent_prices['Volatility_30d_ann'] * 100,
         color='#1a4d3e', lw=1)
ax2.set_title('30-day annualized Volatility (%)')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4 · Bundesbank Svensson Parameters

**Voraussetzung:** Die Bundesbank-CSV muss manuell heruntergeladen werden und unter `data/bundesbank_svensson.csv` liegen.

In [ ]:
%run 04_fetch_bundesbank_svensson.py

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, col in zip(axes.flatten(), svensson_params.columns):
    ax.plot(svensson_params.index, svensson_params[col], lw=0.8)
    ax.set_title(col)
    ax.grid(alpha=0.3)
plt.suptitle('Svensson Parameters · 6 Year History', y=1.02)
plt.tight_layout()
plt.show()

## 5 · Market Proxy (DAX Index)

In [ ]:
%run 05_fetch_market_proxy.py

## 6 · Konsolidierung · `data_layer` Dictionary

Alles in einem Dictionary zusammenführen — das ist die Schnittstelle zum Python-Backend.

In [ ]:
%run 00_build_data_layer.py

# `data_layer` ist jetzt verfügbar im Namespace

In [ ]:
# Quick-Check: was haben wir?
for k, v in data_layer.items():
    if isinstance(v, pd.DataFrame):
        print(f'{k:<25} {v.shape}')
    else:
        print(f'{k:<25} {type(v).__name__}')

## 7 · Data Layer Export

Optional: alles in eine einzige HDF5-Datei speichern, falls du den State zwischen Sessions behalten willst.

In [ ]:
from config import CACHE_DIR
import pickle

state_file = CACHE_DIR / 'data_layer_state.pkl'
with open(state_file, 'wb') as f:
    pickle.dump(data_layer, f)
print(f'✓ data_layer gespeichert: {state_file}')
print(f'  Größe: {state_file.stat().st_size / 1024 / 1024:.1f} MB')

---
## ✅ Data Layer komplett.

**Was jetzt bereit ist:**
- `dax_prices` — 6 Jahre Kurse aller DAX-Mitglieder
- `dax_fundamentals` — Bilanzen + Marktkapitalisierung
- `brent` — Energie-Faktor
- `svensson` — Zinskurven-Parameter + Statistiken
- `market` — Markt-Index für Beta-Schätzung

**Nächster Schritt:** Die Svensson-Engine (`svensson.py`) baut auf diesem Layer auf.